<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/vishaljoshi24/dnd-concordia/main/examples/requirements.in' 'git+https://github.com/vishaljoshi24/dnd-concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
from concordia.utils.structured_logging import AIAgentLogInterface, SimulationLog
log = SimulationLog.from_json(open('simulation_log.json').read())
interface = AIAgentLogInterface(log)

In [ ]:
alice_actions = interface.get_entity_actions('Alice')
bob_actions = interface.get_entity_actions('Bob')

for a in alice_actions:
  print(f"Step {a['step']}: {str(a['action'])[:4]}")

for b in bob_actions:
  print(f"Step {b['step']}: {str(b['action'])[:4]}")

In [ ]:
context = interface.get_entity_action_context('Alice', step=2)
if context:
    print(f"Action: {context['action']}")
    print(f"Observations: {context['observations']}")
    print(f"Prompt: {context['action_prompt']}")

In [ ]:
interface.get_component_values(
    component_key='Observations',
    value_key='Value',
    entity_name='Bob',
    step_range=(1, 4),
)

In [ ]:
for entry in log.entries:
    if entry.entry_type != 'step':
        continue
    full_data = log.reconstruct_value(entry.deduplicated_data)
    resolve = full_data.get('value', {}).get('resolve', {})
    if not resolve:
        continue

    resolution = resolve.get('__resolution__', {})
    if resolution:
        event_text = resolution.get('Value', '')
        print(f"Step {entry.step} event: {event_text[:200]}")

    tracker = resolve.get('tension_tracker', {})
    if tracker:
        act = tracker.get('__act__', tracker)
        tension_val = act.get('Value', '')
        print(f"Step {entry.step} tension: {tension_val}")

In [ ]:

API_TYPE = 'openai'
MODEL_NAME = 'gpt-5'
API_KEY = ''
DISABLE_LANGUAGE_MODEL = False


model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
    api_key=API_KEY
)


DISABLE_LANGUAGE_MODEL = False
if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)



In [ ]:
import re

def ask_evaluation_question(model, question_config, data_str, game_rules,
                            narrative_backstory):
    """Ask an evaluation question and parse the integer score (if applicable)."""

    # Allow config to override the output format instructions for non-scoring tasks
    output_instructions = question_config.get(
        "output_instructions",
        "Output the final score as a single integer. If the question is binary "
        "(Yes/No), output 1 for Yes or 0 for No."
    )
    output_format_template = question_config.get(
        "output_format",
        "Reasoning: [Your step-by-step analysis goes here]\n"
        "FINAL_SCORE: [Integer only]"
    )

    prompt = (
        f"You are an automated judge extracting data from AI simulation logs.\n\n"
        f"### CONTEXT:\n{context}\n\n"
        f"### RULES:\n{game_rules}\n\n"
        f"### TRANSCRIPT:\n{data_str}\n\n"
        f"### QUESTION:\n{question_config['question']}\n\n"
        f"### TASK:\n"
        f"1. Analyze the transcript based on the Question and Rubric.\n"
        f"2. Provide your reasoning.\n"
        f"3. {output_instructions}\n\n"
        f"### OUTPUT FORMAT:\n"
        f"{output_format_template}"
    )

    try:
        response_text = model.sample_text(prompt)
    except Exception as e:
        print(f"Model Generation Error: {e}")
        return {"question_id": question_config['id'], "score": 0, "reasoning": "Error"}

    score = 0
    reasoning = response_text

    # Extract Score using Regex
    match = re.search(r"FINAL_SCORE:\s*(\d+)", response_text)
    if match:
        score = int(match.group(1))
    else:
        # Fallback: Look for the last number
        all_numbers = re.findall(r"(\d+)", response_text)
        if all_numbers:
            score = int(all_numbers[-1])

    # Clean up reasoning
    reasoning = response_text.split("FINAL_SCORE:")[0].replace("Reasoning:", "").strip()

    return {
        "question_id": question_config['id'],
        "score": score,
        "reasoning": reasoning
    }

# --- Usage with AIAgentLogInterface ---
from concordia.utils import structured_logging

# 1. Load the Log using library functions
with open("simulation_log.json") as f:
    log = structured_logging.SimulationLog.from_json(f.read())

interface = structured_logging.AIAgentLogInterface(log)

# 2. Extract Bob's actions
actions = interface.get_entity_actions("Bob")
transcript_str = "\n".join([f"Step {a['step']}: {a['action']}" for a in actions])

context = ('This D&D short scenario specifically concerns a craft brewery.',
'It is set in the Wizards Tower Brewing Company and is in dire need of help.',
'A band of reliable, affordable adventurers are needed to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
'For the duration of the scenario, only Alice, Bob and the brewery owner are in the brewery',
'At the beginning of this adventure Alice and Bob',
'meet in the Wizard Tower Brewing Company. These two adventurers',
'DO NOT know each other AT FIRST and need to get to know each other.',
'The brewery owner hands out pints of Ale to Alice and Bob as they get to know each other.',
)
game_rules = "Alice and Bob take turns in dialog randomly during the scenario."

# Task 1: Tabulate topics discussed
behavioural_realism = {
    "id": "realism",
    "question": " Do agents perform plausible, role appropriate actions for a Dungeons & Dragons player?.",
    "output_instructions": ("Rate on a scale from 1 (poor) to 5 (excellent) each action that an agent takes. Give an aggregate score at the end.",
    "An example of 1/5 on Behavioural Realism is the following action for a Dungeons & Dragons player agent:",
		"Alice responds to the brewery owner\'s offer of 25 gold pieces if she clears the rat infestation by saying:",
    "Nah I think I will go for a holiday on the Sword Coast right now, good luck with the infestation",
	  "An example of 5/5 of Behavioural Realism is the action for a Dungeons & Dragons player agent:",
		"Alice responds to the brewery owner's offer of 25 gold pieces if she clears the rat infestation by saying",
    "We accept this contract to clear out the rats, but for a price! We are but wandering adventures after all and we need to survive."),
    "output_format": "Reasoning: [Your step-by-step analysis goes here]\n"
}

narrative_coherence = {
    "id": "coherence",
    "question": " Do events form logical storylines?.",
    "output_instructions": ("Rate on a scale from 1 (poor) to 5 (excellent) each action that an agent takes. Give an aggregate score at the end.",
    "An example of 1/5 on Narrative Coherence is the following exchange between two  Dungeons & Dragons player agents:",
    "Bob, when asked if he can check the basement first to scope for rats as he has dark vision responds angrily,",
    "'No I do not want to do that, why don't you do it you lazy bum, I am leaving! You can have all the money for yourself I don't care.'",
    "As a result the scenario is totally abandoned as Alice will not take on the rats by herself.",
    "An example of 5/5 on Narrative Coherence is the following action for a Dungeons & Dragons player agent:",
    "Bob, when asked if he can check the basement first to scope for rats as he has dark vision responds:",
    "'That is one way of doing it but I could also provide cover fire with my crossbow, if you want to head down with a torch?'",
    "Alice thinks on this and likes the idea, going down first with Bob providing cover fire."),
    "output_format": "Reasoning: [Your step-by-step analysis goes here]\n"
}

llm_groundedness = {
    "id": "groundedness",
    "question": "Do events stay within simulation reality?.",
    "output_instructions": ("Rate on a scale from 1 (poor) to 5 (excellent) each action that an agent takes. Give an aggregate score at the end.",
    "An example of 1/5 on LLM Groundedness is the following scenario in a Dungeons & Dragons context:",
	  "Alice is waiting in a craft brewery in a city that is in the Forgotten Realms setting, for another adventurer to come along",
    "and help her clean out a rat infestation at the brewery. She decides to use her phone and call up Bob who says",
    "that he is in Somalia at the moment. An example of 5/5 on LLM Groundedness is the following scenario in a Dungeons & Dragons context:",
		"Alice is waiting in the craft brewery, which is set in the city of Neverwinter, for another adventurer to come along and help with the rat infestation at the brewery.",
    "She is sipping her pint of ale waiting for this other individual when a",
    "strange bard walks in the door and introduces himself. His name is Bob and he is here about the rat infestation."),
    "output_format": "Reasoning: [Your step-by-step analysis goes here]\n"
}

agent_distinctiveness = {
    "id": "distinctiveness",
    "question": "Do agents feel like unique individuals?.",
    "output_instructions": ("Rate on a scale from 1 (poor) to 5 (excellent) each action that an agent takes. Give an aggregate score at the end.",
    "An example of 1/5 on Agent Distinctiveness is the following exchange between agents in a Dungeons & Dragons scenario:",
		"Alice, when talking about her origins of why she became an adventurer, expresses sorrow at her family dying when she was quite young and having to",
    "be self sufficient and resilient, forcing her to turn to a life of crime where she was brought up in a criminal gang.",
		"Bob when talking about his origins says that his parents died and he fell in with a group of adventurers who took care of him and raised him.",
	  "An example of 5/5 on Agent Distinctiveness is the following exchange between agents in a Dungeons & Dragons scenario:",
		"Alice, when talking about her origins as an adventurer says - 'I was captive of a dragon in the mountains which had killed my people.",
    "'In slaying the dragon and escaping imprisonment I found meaning and realised that I wanted to help those in need, just like me.'",
		"Bob says - 'I was quite bored of my job as government clerk in the frozen north, both because of the dreary environment and the really bad beer!'",
    "'I thought adventuring would be a more exciting job and give me the opportunity sample beers from around the Forgotten Realms.'"),
    "output_format": "Reasoning: [Your step-by-step analysis goes here]\n"
}




realism_result = ask_evaluation_question(
    model, behavioural_realism, transcript_str, game_rules, context
)
print("### Realism (from Reasoning):")
print(realism_result["reasoning"])

coherence_result = ask_evaluation_question(
    model, narrative_coherence, transcript_str, game_rules, context
)
print("### Coherence (from Reasoning):")
print(coherence_result["reasoning"])

groundedness_result = ask_evaluation_question(
    model, llm_groundedness, transcript_str, game_rules, context
)
print("### Groundedness (from Reasoning):")
print(groundedness_result["reasoning"])

distinctiveness_result = ask_evaluation_question(
    model, agent_distinctiveness, transcript_str, game_rules, context
)
print("### Distinctiveness (from Reasoning):")
print(distinctiveness_result["reasoning"])
